# Zadania: SAM na ortofotomapie z Łomianek

Zestaw ćwiczeń do **poziomu 2** (wykład: `02_segmentacja_zero_shot_sam.md`, 20 slajdów). Na wykładzie SAM pracuje na ortofotomapie z Tapias (zabudowa, dachy, drogi).
Tutaj robisz to samo **na innym zdjęciu**: `dane/lomianki_ortofoto.tif`: łąki, ściernisko, kępy drzew i krzewów, droga gruntowa, **bez zabudowy**
(1700 × 1700 px, piksel 0,1 m, EPSG:32634).

**Ten notebook jest prawie pusty celowo.** Gotowe są tylko komórki oznaczone „Gotowe” (importy, wczytanie zdjęcia, rysowanie oraz pełne funkcje kafelkowania i atrybutów,
których slajdy nie pokazują w całości). Resztę **kopiujesz z wykładu**:

1. znajdź slajd wskazany w zadaniu i skopiuj z niego blok kodu do pustej komórki,
2. zmień to, co pochodzi z Tapias (punkty, współrzędne, rozmiary), i uruchom,
3. odpowiedz na pytania pod zadaniem, patrząc na wynik.

Zadania idą po kolei i korzystają ze zmiennych z poprzednich (`sam`, `predictor`, `segmentuj_punkty`, `obiekty_z_sam` …). Pliki, które zapiszesz, trafiają do `wyniki/zadania_sam/`.
Obliczenia idą na CPU: klik to ułamek sekundy, tryb automatyczny od kilku sekund do około minuty.

| Zadanie | Temat | Slajd wykładu | Co kopiujesz |
|---|---|---|---|
| 1 | Model SAM i skala | 5 | wczytanie `sam` |
| 2 | Klik w punkt | 6 | `SamPredictor`, `segmentuj_punkty` |
| 3 | Trzy maski z jednego kliknięcia | 7 | nic (rysujesz gotową `pokaz_maski`) |
| 4 | Punkty pozytywne i negatywne | 8 | nic (`segmentuj_punkty` z zadania 2) |
| 5 | Klik we współrzędnych mapy | 9, 10 | `mapa_na_piksele`, `maska_do_poligonu` |
| 6 | Automatyczne maski na kadrze | 11, 12, 13 | generator, `obiekty_z_sam` |
| 7 | Filtry generatora | 11 | nic (generator z zadania 6) |
| 8 | Całe zdjęcie naraz | 14 | nic (generator z zadania 6) |
| 9 | Kafelkowanie | 15, 16, 17 | nic (gotowa `maski_kafelkami`) |
| 10 | Atrybuty i klasa „zielone” | 18 | nic (gotowa `obiekty_do_gdf`) |
| 11 | Eksport do GeoPackage i QGIS | 19 | zapis GPKG (geopandas) |

## Część 0. Gotowe: uruchom po kolei, bez zmian

In [ ]:
!uv sync

In [ ]:
import time
import urllib.request
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from affine import Affine
from rasterio.features import shapes
from segment_anything import SamAutomaticMaskGenerator, SamPredictor, sam_model_registry
from shapely.geometry import shape
from shapely.ops import unary_union

plt.rcParams.update({"figure.dpi": 90, "axes.titlesize": 11})

SCIEZKA_ZDJECIA = Path("data/lomianki_ortofoto.tif")
KATALOG_WYNIKOW = Path("results/zadania_sam")
KATALOG_WYNIKOW.mkdir(parents=True, exist_ok=True)

# Wagi modelu vit_b (375 MB): pobierają się jednorazowo, jeśli ich jeszcze nie ma. Kod ze slajdu 5 wczytuje je stąd.
PLIK_WAG = Path("modele/sam_vit_b_01ec64.pth")
if not PLIK_WAG.exists():
    PLIK_WAG.parent.mkdir(exist_ok=True)
    print(f"Pobieram {PLIK_WAG.name} (jednorazowo)...")
    urllib.request.urlretrieve("https://dl.fbaipublicfiles.com/segment_anything/" + PLIK_WAG.name, PLIK_WAG.with_suffix(".part"))
    PLIK_WAG.with_suffix(".part").replace(PLIK_WAG)

# Kadr do klikania (wiersze, kolumny): kępa drzew z pasem żółtej nawłoci, oraz jego położenie w całym zdjęciu
FRAGMENT = np.s_[1250:1650, 550:950]
Y0, X0 = FRAGMENT[0].start, FRAGMENT[1].start
print("Wagi modelu:", PLIK_WAG, "| kadr: x od", X0, ", y od", Y0)

In [ ]:
def wczytaj_zdjecie(sciezka):
    """Zwraca: rgb (H,W,3 uint8), waznie (H,W bool), profil (transform, crs), gsd [m/piksel]."""
    with rasterio.open(sciezka) as src:
        rgb = np.moveaxis(src.read([1, 2, 3]), 0, -1)
        profil = {"transform": src.transform, "crs": src.crs}
        gsd = src.res[0]
    waznie = ~(rgb == 0).all(axis=-1)          # czarne tło ortomozaiku = brak danych
    return rgb, waznie, profil, gsd


rgb, waznie, profil, gsd = wczytaj_zdjecie(SCIEZKA_ZDJECIA)
kadr = rgb[FRAGMENT].copy()                                         # kadr jako osobna tablica (H, W, 3)
KADR_EXTENT = (X0, X0 + kadr.shape[1], Y0 + kadr.shape[0], Y0)      # osie wykresów = piksele CAŁEGO zdjęcia

print(f"Zdjęcie: {rgb.shape[1]} x {rgb.shape[0]} px, GSD {gsd:.2f} m, układ {profil['crs']}")
print(f"Kadr do klikania: {kadr.shape[1]} x {kadr.shape[0]} px")

fig, ax = plt.subplots(1, 2, figsize=(14, 6.8))
ax[0].imshow(rgb)
ax[0].add_patch(plt.Rectangle((X0, Y0), kadr.shape[1], kadr.shape[0], fill=False, ec="cyan", lw=2))
ax[0].set_title("Zdjęcie i kadr do klikania (ramka)"); ax[0].axis("off")
ax[1].imshow(kadr, extent=KADR_EXTENT)
ax[1].set_xticks(range(X0, X0 + 401, 50)); ax[1].set_yticks(range(Y0, Y0 + 401, 50))
ax[1].grid(color="yellow", alpha=0.5, lw=0.6); ax[1].tick_params(labelsize=8)
ax[1].set_title("Kadr: osie to piksele CAŁEGO zdjęcia (x = kolumna, y = wiersz)")
plt.tight_layout()

In [ ]:
def nakladka(rgb, maska, kolor=(255, 0, 255), alfa=0.55):
    """Półprzezroczysta maska nałożona na zdjęcie (do oceny wzrokowej)."""
    wynik = rgb.astype(np.float32)
    wynik[maska] = (1 - alfa) * wynik[maska] + alfa * np.array(kolor, dtype=np.float32)
    return wynik.astype(np.uint8)


def pokaz_maski(ax, obraz, maska, punkty=(), etykiety=(), tytul=""):
    """Maska na kadrze (osie = piksele całego zdjęcia); punkty: zielona gwiazdka = pozytywny, czerwona = negatywny."""
    ax.imshow(nakladka(obraz, maska), extent=KADR_EXTENT)
    for (x, y), e in zip(punkty, etykiety):
        ax.plot(x, y, "*", ms=15, mec="k", color="lime" if e else "red")
    ax.set_title(tytul); ax.axis("off")


def koloruj_obiekty(obraz, obiekty, alfa=0.6, seed=1):
    """Każdy obiekt w innym losowym kolorze. Większe maluje pierwsze, mniejsze na wierzchu, bo maski się nakładają.
    Obiekt to słownik z kluczami: maska (przycięta do ramki), wiersz, kolumna, piksele (tak jak zwraca `obiekty_z_sam`)."""
    rng = np.random.default_rng(seed)
    wynik = obraz.astype(np.float32)
    for o in sorted(obiekty, key=lambda o: -o["piksele"]):
        wys, szer = o["maska"].shape
        okno = wynik[o["wiersz"]:o["wiersz"] + wys, o["kolumna"]:o["kolumna"] + szer]
        okno[o["maska"]] = (1 - alfa) * okno[o["maska"]] + alfa * rng.integers(40, 255, 3)
    return wynik.astype(np.uint8)


def czesc_wazna(obiekt, waznie):
    """Jaka część pikseli obiektu leży na ważnych danych (a nie na czarnym tle ortomozaiku)."""
    wys, szer = obiekt["maska"].shape
    return waznie[obiekt["wiersz"]:obiekt["wiersz"] + wys, obiekt["kolumna"]:obiekt["kolumna"] + szer][obiekt["maska"]].mean()


print("Gotowe: rgb, waznie, profil, gsd, kadr, X0, Y0 | funkcje: nakladka, pokaz_maski, koloruj_obiekty, czesc_wazna")

## Część A. Model i klik w punkt

### Zadanie 1. Model SAM i skala

Wczytaj model `vit_b` do zmiennej `sam` (**slajd 5**, jedna linijka; wagi masz już w `modele/`).
Potem policz skalę: SAM skaluje **dłuższy bok** obrazu do 1024 px. Ile razy zmniejsza się przy tym całe zdjęcie, a ile razy powiększa się kadr (400 px)?
Ile centymetrów ma piksel po skalowaniu w obu przypadkach?

> **Podpowiedź.** Piksel po skalowaniu = `gsd × dłuższy_bok / 1024`. Slajd 5 podaje te liczby dla Tapias.

In [ ]:
# slajd 5: wczytaj model do zmiennej sam

# skala: dla rgb (całe zdjęcie) i dla kadru

### Zadanie 2. Klik w punkt

Skopiuj ze **slajdu 6** cały blok: `SamPredictor`, `set_image` i funkcję `segmentuj_punkty`. Blok kończy się kliknięciem w punkt `(1210, 1075)`.
To punkt z **Tapias**: na Łomiankach leży poza kadrem. Najpierw uruchom blok **bez zmian** i zobacz, co zwraca `predict`, a potem podmień punkt na środek **kępy drzew**, około `(750, 1340)`.
Dla każdej z trzech masek wypisz score i powierzchnię w m² (`maska.sum() * gsd ** 2`).

**Pytania:**
- (a) Czy punkt spoza kadru wywołał błąd? Co to mówi o kontroli poprawności danych wejściowych w SAM?
- (b) Punkt to `(x, y)` = `(kolumna, wiersz)`. Która z trzech masek pokrywa całą kępę, a która tylko jej fragment?

> **Podpowiedź.** Punkt podajesz w pikselach **całego zdjęcia**. Funkcja sama odejmuje `(X0, Y0)`. Kępę drzew znajdziesz na siatce kadru powyżej.

In [ ]:
# slajd 6: predictor = SamPredictor(sam) ... segmentuj_punkty(...)
# 1. uruchom blok bez zmian (punkt z Tapias), obejrzyj wynik
# 2. podmień punkt na kępę drzew, wypisz score i powierzchnię każdej z 3 masek

### Zadanie 3. Trzy maski z jednego kliknięcia

Wywołaj `segmentuj_punkty(..., wiele_masek=True)` osobno dla kępy drzew `(750, 1340)` i dla pasa żółtej nawłoci `(880, 1480)`.
Narysuj wyniki w dwóch rzędach po trzy maski (gotowa `pokaz_maski`; układ jak na **slajdzie 7**) z tytułem: numer maski, score i powierzchnia w m².

**Pytania:** Czy maska o najwyższym score jest zawsze tą, o którą Ci chodziło? Kiedy tak, a kiedy nie?

> **Podpowiedź.** `fig, ax = plt.subplots(2, 3, figsize=(13, 8.8))`, potem `pokaz_maski(ax[w, k], kadr, maski[k], [punkt], [1], "tytuł")`.

In [ ]:
# PRZYKLADY = {"kępa drzew": (750, 1340), "nawłoć": (880, 1480)}
# fig, ax = plt.subplots(2, 3, ...)
# w pętli: maski, jakosci = segmentuj_punkty([punkt], wiele_masek=True), a potem pokaz_maski(...) dla k = 0, 1, 2

### Zadanie 4. Punkty pozytywne i negatywne

Dołóż drugi punkt i sprawdź, jak zmienia się maska kępy drzew. Trzy warianty (**slajd 8**): sam punkt na kępie, kępa + punkt **pozytywny** na nawłoci `(880, 1480)`, kępa + punkt **negatywny** w tym samym miejscu.
Przy kilku punktach proś o **jedną** maskę (`wiele_masek=False`). Narysuj trzy warianty obok siebie i podpisz powierzchnię oraz score.

**Pytania:** Co robi punkt pozytywny, a co negatywny z maską? Ile punktów według Ciebie wystarcza?

> **Podpowiedź.** Etykiety: `1` = pozytywny, `0` = negatywny, np. `segmentuj_punkty([DRZEWA, NAWLOC], [1, 0])`. Żeby zobaczyć różnicę z bliska: `a.set_xlim(600, 950); a.set_ylim(1580, 1250)`.

In [ ]:
# DRZEWA = (750, 1340); NAWLOC = (880, 1480)
# warianty = [("1 punkt", [DRZEWA], [1]), ("+ pozytywny", ...), ("+ negatywny", ...)]
# fig, ax = plt.subplots(1, 3, ...), w pętli: segmentuj_punkty(punkty, etykiety) i pokaz_maski(...)

### Zadanie 5. Klik we współrzędnych mapy

Skopiuj ze **slajdu 9** funkcje `mapa_na_piksele` i `maska_do_poligonu`, obliczenie `transform_kadru` oraz kliknięcie w punkt podany w metrach.
Współrzędne z Tapias (`461123.2, 492317.1`) tu nie zadziałają, bo zdjęcie jest w **EPSG:32634**. Użyj `(490661.9, 5801753.5)` (kępa drzew) i `(490674.9, 5801739.5)` (pas nawłoci)
albo odczytaj własne w QGIS-ie (otwórz `dane/lomianki_ortofoto.tif`).
Wyniki obu kliknięć zbierz w GeoDataFrame `klikniete` (kolumny `nazwa`, `pow_m2`, `score`, `geometry`; `crs=profil["crs"]`). Przyda się w zadaniu 11.

**Pytania:**
- (a) Na jakie piksele (`kolumna, wiersz`) przeliczają się te współrzędne? Zgadzają się z punktami z zadań 2–4?
- (b) Pułapka ze slajdu 9: zbuduj poligon kępy z `profil["transform"]` (transformacja **całego zdjęcia**) zamiast `transform_kadru`. Porównaj `bounds` obu poligonów. Który leży na kępie?

> **Podpowiedź.** `GeoDataFrame(lista_słowników, geometry="geometry", crs=profil["crs"])`. Powierzchnia wyjdzie dobrze w obu wersjach poligonu, a położenie tylko w jednej.

In [ ]:
# slajd 9: mapa_na_piksele, maska_do_poligonu, transform_kadru, kliknięcie w (E, N)
# współrzędne: kępa (490661.9, 5801753.5), nawłoć (490674.9, 5801739.5)
# klikniete = gpd.GeoDataFrame([...], geometry="geometry", crs=profil["crs"])

## Część B. Automatyczne maski

### Zadanie 6. Automatyczne maski na kadrze

Skopiuj ze **slajdu 12** generator `SamAutomaticMaskGenerator` i funkcję `obiekty_z_sam`. Uwaga: na slajdzie `obiekty_z_sam` jest **generatorem Pythona** (`yield`), więc zamień wynik na listę,
`obiekty = list(obiekty_z_sam(maski_sam))`. Wtedy zadziała `len(...)` i użyjesz go więcej niż raz.
Uruchom dla siatki `points_per_side` = **16**, potem **32** (filtry domyślne). Zmierz czas i liczbę masek, a wynik narysuj przez `koloruj_obiekty(kadr, obiekty)` (jak na **slajdzie 13**).

**Pytania:**
- (a) Ile masek i ile sekund dla każdej siatki? Jak rośnie czas przy podwojeniu `points_per_side`?
- (b) Które obiekty kadru znalazł SAM, a których brakuje? Czy jest kępa drzew z zadań 2–4?

> **Podpowiedź.** Czas: `t0 = time.perf_counter()` przed `generate`. Wynik `generate` to lista słowników (`segmentation`, `area`, `predicted_iou` …).

In [ ]:
# slajd 12: generator = SamAutomaticMaskGenerator(...); obiekty_z_sam
# w pętli po points_per_side in (16, 32): czas, maski_sam = generator.generate(kadr), obiekty = list(obiekty_z_sam(maski_sam))
# rysunek: koloruj_obiekty(kadr, obiekty)

### Zadanie 7. Filtry generatora

Tabela na **slajdzie 11** opisuje parametry, które **odsiewają** maski. Na Łomiankach domyślne progi bywają za ostre.
Uruchom generator 32 × 32 z `stability_score_thresh=0.80` (zamiast domyślnego 0,95) i porównaj z generatorem domyślnym: liczba masek i to, co się pojawiło na rysunku.
Potem sprawdź maski, które **obejmują punkt w kępie** `(750, 1340)`: wypisz ich powierzchnię (m²), `predicted_iou` i `stability_score`.
Żeby zobaczyć też maski odrzucone, uruchom jeszcze generator z **wyłączonymi** filtrami: `pred_iou_thresh=0, stability_score_thresh=0`.

**Pytania:**
- (a) Który filtr odrzuca maskę całej kępy przy ustawieniach domyślnych i o ile jej ocena jest za niska?
- (b) Co znika, gdy podniesiesz progi (`pred_iou_thresh=0.95`, `stability_score_thresh=0.98`)?

> **Podpowiedź.** Czy maska obejmuje punkt `(x, y)`: `d["segmentation"][y - Y0, x - X0]` (współrzędne kadru = współrzędne zdjęcia minus `(X0, Y0)`).

In [ ]:
# generator z parametrami: SamAutomaticMaskGenerator(sam, points_per_side=32, min_mask_region_area=100, stability_score_thresh=0.80)
# a) domyślny vs 0.80: liczba masek, rysunek
# b) filtry wyłączone (pred_iou_thresh=0, stability_score_thresh=0): tabela masek obejmujących punkt kępy

### Zadanie 8. Całe zdjęcie naraz

Puść generator (32 × 32, `stability_score_thresh=0.80`) na **całym** zdjęciu `rgb` (**slajd 14**). Zmierz czas, policz maski i pamięć pełnowymiarowych masek
(`d["segmentation"].nbytes` sumowane po masce), a wynik narysuj obok zdjęcia.

**Pytania:**
- (a) Ile masek, ile sekund i ile MB? Jak by to wyglądało dla ortofotomapy 10 000 × 10 000 px?
- (b) Ile masek leży w większości na czarnym tle poza mozaiką (`czesc_wazna(o, waznie) < 0.5`)?
- (c) Które obiekty SAM znalazł na całym zdjęciu, a które gubi przy skali 1,7×?

> **Podpowiedź.** `obiekty_cale = list(obiekty_z_sam(maski_cale))` daje obiekty gotowe do `koloruj_obiekty(rgb, ...)` i `czesc_wazna`.

In [ ]:
# generator 32 x 32 (stability_score_thresh=0.80) na rgb; czas, len(maski), pamięć w MB
# obiekty_cale = list(obiekty_z_sam(maski_cale)); maski na czarnym tle; rysunek zdjęcie | maski

## Część C. Kafelki i obiekty GIS

### Gotowe: pełna funkcja kafelkowania

Kod na **slajdzie 16** jest skrócony (nie definiuje `dotyka_wewnetrznej_krawedzi` ani listy `kandydaci`), więc nie da się go uruchomić po samym skopiowaniu.
Poniżej pełna, działająca wersja tej funkcji wraz z pomocniczymi `poczatki_kafli` i `iou_obiektow`. Uruchom komórkę bez zmian.
Korzysta z `obiekty_z_sam` i `czesc_wazna`, więc zadanie 6 musi być zrobione.

In [ ]:
def poczatki_kafli(n, kafel, zakladka):
    """Początki kafli o boku `kafel` px, równomiernie pokrywających `n` px, z zakładką nie mniejszą niż `zakladka`."""
    if n <= kafel:
        return [0]
    liczba = int(np.ceil((n - zakladka) / (kafel - zakladka)))
    return np.linspace(0, n - kafel, liczba).round().astype(int).tolist()


def iou_obiektow(a, b):
    """IoU dwóch obiektów zapisanych jako przycięte maski z położeniem (bez budowania pełnowymiarowych masek)."""
    w0, k0 = max(a["wiersz"], b["wiersz"]), max(a["kolumna"], b["kolumna"])
    w1 = min(a["wiersz"] + a["maska"].shape[0], b["wiersz"] + b["maska"].shape[0])
    k1 = min(a["kolumna"] + a["maska"].shape[1], b["kolumna"] + b["maska"].shape[1])
    if w1 <= w0 or k1 <= k0:
        return 0.0
    ma = a["maska"][w0 - a["wiersz"]:w1 - a["wiersz"], k0 - a["kolumna"]:k1 - a["kolumna"]]
    mb = b["maska"][w0 - b["wiersz"]:w1 - b["wiersz"], k0 - b["kolumna"]:k1 - b["kolumna"]]
    wspolna = (ma & mb).sum()
    return wspolna / (a["piksele"] + b["piksele"] - wspolna)


def maski_kafelkami(rgb, waznie, generator, gsd, kafel_m=77, zakladka_m=15, min_obiekt_m2=2.0):
    """SAM na kafelkach z zakładką. Zwraca listę obiektów (przycięta maska + położenie w całym zdjęciu)."""
    wys_zdj, szer_zdj = waznie.shape
    kafel, zakladka = round(kafel_m / gsd), round(zakladka_m / gsd)
    kandydaci = []
    for w0 in poczatki_kafli(wys_zdj, kafel, zakladka):
        for k0 in poczatki_kafli(szer_zdj, kafel, zakladka):
            okno = np.s_[w0:w0 + kafel, k0:k0 + kafel]
            if waznie[okno].mean() < 0.05:                       # kafel prawie bez danych
                continue
            wys, szer = waznie[okno].shape
            for o in obiekty_z_sam(generator.generate(rgb[okno].copy()), w0, k0):
                h, w = o["maska"].shape
                y, x = o["wiersz"] - w0, o["kolumna"] - k0        # położenie maski w kaflu
                ucieta = ((x == 0 and k0 > 0) or (y == 0 and w0 > 0)                       # dotyka wewnętrznej krawędzi kafla
                          or (x + w == szer and k0 + szer < szer_zdj) or (y + h == wys and w0 + wys < wys_zdj))
                if not ucieta and czesc_wazna(o, waznie) >= 0.5 and o["piksele"] * gsd ** 2 >= min_obiekt_m2:
                    kandydaci.append(o)

    kandydaci.sort(key=lambda o: -o["iou"])                     # najpewniejsze pierwsze
    zostawione = []
    for o in kandydaci:
        if all(iou_obiektow(o, z) < 0.7 for z in zostawione):    # duplikat z zakładki?
            zostawione.append(o)
    return zostawione


print("Gotowe: poczatki_kafli, iou_obiektow, maski_kafelkami")

### Zadanie 9. Kafelkowanie

Uruchom `maski_kafelkami(rgb, waznie, generator, gsd, kafel_m=...)` (**slajdy 15–17**) z generatorem 16 × 16 punktów i `stability_score_thresh=0.80`.
Zapisz wynik jako `obiekty_kafle` i porównaj **dwa rozmiary kafla**: `kafel_m=60` i `kafel_m=120`. Dla każdego zmierz czas i liczbę obiektów oraz narysuj wynik (`koloruj_obiekty(rgb, ...)`).
Dla rozmiaru 120 m zapisz wynik jako `obiekty_kafle` (będzie potrzebny w zadaniach 10 i 11).

**Pytania:**
- (a) Które duże obiekty (ściernisko, łąka, droga) znikają przy małym kaflu, a które przy dużym?
- (b) Dlaczego obiekt większy niż kafel nie może się pojawić w wyniku? Jaki rozmiar kafla wybierzesz dla tego zdjęcia?
- (c) Porównaj z wynikiem dla całego zdjęcia z zadania 8: liczba obiektów, mediana powierzchni (m²).

> **Podpowiedź.** Kafle 60 m mają 600 px, a kafle 120 m aż 1200 px, więc pamiętaj, że SAM i tak skaluje każdy kafel do 1024 px. Kafel 60 m liczy się dłużej (ok. minuty).

In [ ]:
# generator = SamAutomaticMaskGenerator(sam, points_per_side=16, min_mask_region_area=100, stability_score_thresh=0.80)
# w pętli po kafel_m in (60, 120): czas, obiekty = maski_kafelkami(rgb, waznie, generator, gsd, kafel_m=kafel_m)
# obiekty_kafle = wynik dla 120 m; rysunek: koloruj_obiekty(rgb, obiekty)

### Gotowe: funkcja z atrybutami

Slajd 18 pokazuje tylko trzy linijki liczenia atrybutów. Poniżej pełna funkcja `obiekty_do_gdf`, która zamienia listę obiektów SAM-a na GeoDataFrame:
poligon w układzie zdjęcia, `pow_m2`, `zwartosc`, średni `exg_sr`, klasa `zielone` (ExG > 0,10, reguła z poziomu 1) i oceny `sam_iou`, `sam_stabilnosc`.
Korzysta z `maska_do_poligonu` z zadania 5.

In [ ]:
def exg(rgb):
    """Excess Green z współrzędnych chromatycznych, jak w poziomie 1."""
    R, G, B = (rgb[..., i].astype(np.float32) for i in range(3))
    suma = R + G + B + 1e-6
    return 2 * G / suma - R / suma - B / suma


PROG_EXG = 0.10      # obiekt jest „zielony", gdy jego średni ExG jest wyższy


def obiekty_do_gdf(obiekty, rgb, profil, gsd):
    """Obiekty SAM-a -> GeoDataFrame: poligon w układzie zdjęcia + atrybuty."""
    mapa_exg = exg(rgb)
    wiersze = []
    for i, o in enumerate(obiekty, start=1):
        wys, szer = o["maska"].shape
        exg_sr = float(mapa_exg[o["wiersz"]:o["wiersz"] + wys, o["kolumna"]:o["kolumna"] + szer][o["maska"]].mean())
        geometria = maska_do_poligonu(o["maska"], profil["transform"] * Affine.translation(o["kolumna"], o["wiersz"]), gsd)
        wiersze.append({
            "id": i,
            "pow_m2": round(o["piksele"] * gsd ** 2, 1),
            "zwartosc": round(4 * np.pi * geometria.area / geometria.length ** 2, 2),
            "exg_sr": round(exg_sr, 3),
            "zielone": exg_sr > PROG_EXG,
            "sam_iou": round(o["iou"], 2),
            "sam_stabilnosc": round(o["stabilnosc"], 2),
            "geometry": geometria,
        })
    return gpd.GeoDataFrame(wiersze, geometry="geometry", crs=profil["crs"])


print("Gotowe: exg, PROG_EXG, obiekty_do_gdf")

### Zadanie 10. Atrybuty i klasa „zielone”

Zamień `obiekty_kafle` na GeoDataFrame: `obiekty_gdf = obiekty_do_gdf(obiekty_kafle, rgb, profil, gsd)` (**slajd 18**).
Wypisz liczbę obiektów i liczbę „zielonych”. Pokaż osiem największych obiektów (`pow_m2`, `zwartosc`, `exg_sr`, `zielone`).
Narysuj obiekty na zdjęciu: zielone `limegreen`, reszta `orange`. Na koniec zastosuj filtr z wykładu: `pow_m2` od 15 do 300 i `zwartosc > 0.6`.

**Pytania:**
- (a) Ile obiektów jest „zielonych”? Które z **niezielonych** to ściernisko, droga lub cień i jaki mają `exg_sr`?
- (b) Ile obiektów przechodzi filtr `15–300 m²` i `zwartosc > 0.6` i co jeszcze się przedostało? Na Tapias filtr wskazywał dachy. Co wskazuje tutaj?
- (c) Na jednym obiekcie sprawdź, że `pow_m2` to liczba pikseli maski razy `gsd ** 2`.

> **Podpowiedź.** Rysunek z podziałem na dwa kolory: `gdf.plot(ax=ax, color=np.where(gdf.zielone, "limegreen", "orange"), alpha=0.5)`. Przy `column="zielone"` geopandas maluje wszystko jednym kolorem, gdy w danych jest tylko jedna wartość.

In [ ]:
# obiekty_gdf = obiekty_do_gdf(obiekty_kafle, rgb, profil, gsd)
# print: len, zielone.sum(); obiekty_gdf.drop(columns="geometry").sort_values("pow_m2", ascending=False).head(8)
# rysunek: zdjęcie z extent = (lewo, prawo, dół, góra) z profil["transform"], potem obiekty_gdf.plot(ax=ax, color=...)
# filtr: obiekty_gdf[(obiekty_gdf.pow_m2.between(15, 300)) & (obiekty_gdf.zwartosc > 0.6)]

### Zadanie 11. Eksport do GeoPackage i QGIS

Zapisz `obiekty_gdf` do `wyniki/zadania_sam/sam_obiekty.gpkg` jako warstwę `automatyczne` (**slajd 19**). Jeśli zadanie 5 jest zrobione, dodaj do **tego samego pliku** warstwę `z_kliku` z `klikniete`.
Wypisz warstwy pliku (`gpd.list_layers`). Otwórz plik w QGIS-ie razem z `dane/lomianki_ortofoto.tif` i ostyluj warstwę `automatyczne` po polu `zielone`.
Na koniec zsumuj `pow_m2` wszystkich obiektów i porównaj z powierzchnią całego zdjęcia w m² (piksele × `gsd ** 2`).

**Pytanie:** Dlaczego suma `pow_m2` nie jest pokryciem terenu i czemu nie wolno jej porównywać z procentami roślinności z poziomu 1?

> **Podpowiedź.** `gdf.to_file(ścieżka, layer="nazwa", driver="GPKG")`. Przed zapisem usuń stary plik: `ścieżka.unlink(missing_ok=True)`. W QGIS-ie: *Warstwa → Dodaj warstwę → Dodaj warstwę wektorową* i wybierz plik `.gpkg`.

In [ ]:
# sciezka = KATALOG_WYNIKOW / "sam_obiekty.gpkg"; sciezka.unlink(missing_ok=True)
# obiekty_gdf.to_file(sciezka, layer="automatyczne", driver="GPKG"); to samo dla klikniete -> "z_kliku"
# print(gpd.list_layers(sciezka))
# suma pow_m2 vs powierzchnia zdjęcia w m2

## Dla chętnych

- Zmień `vit_b` na `vit_l` (wagi ok. 1,2 GB, adres i nazwa pliku jak w komórce startowej: `sam_vit_l_0b3195.pth`). Czy poprawia to maski kępy drzew i o ile wydłuża czas?
- Weź własne zdjęcie z drona (GeoTIFF w układzie metrycznym) i powtórz zadania 6–11. Co zmieni się w doborze `kafel_m` i filtrów?
- Przerzuć wszystkie kroki zadań 8–11 do jednej funkcji `segmentuj(sciezka)` (slajd 19 nazywa ją `segmentuj_sam`) i puść ją na kilku zdjęciach.